# Challenge #4: Date Parsing

> For the fourth challenge let’s look at parsing Dates from text strings. To view the previous challenge, click HERE.

>A dataset contains a text field that has a date embedded within the text. The problem is that the date is represented a few different ways. For example:
```
16-APR-2005
Nov•16,•1900
4-SEP-00
Jan•5•2000
```
>The goal is to **create a new Date/Time field populated with the dates contained within the text field. You will also need to standardize the dates so that they are all formatted the same.**

>We have listed this as an advanced exercise since parsing out the dates can be challenging depending on the technique you employ to do it. As always, we love to hear your comments. Have fun!

### Sample Output:

In [ ]:
import pandas as pd
import numpy as np

# import and preview output

# quoting=3 to specify no quote handling
# otherwise the last five rows all merge together because of an unclosed quotation mark
output = pd.read_csv('output.csv', index_col = False, sep='|', quoting=3)
df_output = pd.DataFrame(output)
df_output.head(25)

,Field1,DateTime_out
0,He who sleeps on the floor will not fall off t...,2005-04-16
1,"After all is said and done, more is said than ...",1856-01-09
2,I want to see you shoot the way you shoutTeddy...,1900-11-16
3,get someone else to do it.15-APR-1944This reco...,1944-04-15
4,Why do they call it rush hour when nothing mov...,1970-06-27
5,I'm taking the Ryanair approach to it: subcont...,2011-05-23
6,I Xeroxed a mirror. Now I have an extra Xerox...,2006-06-30
7,Freidrich Engels01-AUG-08This record is missin...,2008-08-01
8,'He's so old his social security number is two...,2000-01-05
9,"""I was the best man at the wedding.So why is s...",2001-07-09


### Input:

In [ ]:
# import and preview input

input = pd.read_csv("input.csv", index_col=False, sep="|", quoting=3)
df = pd.DataFrame(input)

# you can extend the view if you want

# max_str = df.Field1.str.len().max()
# pd.options.display.max_colwidth = max_str

df

,Field1
0,He who sleeps on the floor will not fall off t...
1,"After all is said and done, more is said than ..."
2,I want to see you shoot the way you shoutTeddy...
3,get someone else to do it.15-APR-1944This reco...
4,Why do they call it rush hour when nothing mov...
5,I'm taking the Ryanair approach to it: subcont...
6,I Xeroxed a mirror. Now I have an extra Xerox...
7,Freidrich Engels01-AUG-08This record is missin...
8,'He's so old his social security number is two...
9,"""I was the best man at the wedding.So why is s..."


In [ ]:
# 16-APR-2005
# Nov 16, 1900
# 4-SEP-00
# Jan 5 2000

# strings formatted as 11-JAN
df["day_month"] = df.Field1.str.extract(r"(\d{1,2}[-\s][a-zA-Z]{3,4}),*[-\s]\d{2,4}")
# strings formatted as JAN 11
df["month_day"] = df.Field1.str.extract(r"([a-zA-Z]{3}[-\s]\d{1,2}),*[-\s]\d{2,4}")
# strings formatted either way
df["year"] = df.Field1.str.extract(r"(?:\d{1,2}[-\s][a-zA-Z]{3,4}|[a-zA-Z]{3}[-\s]\d{1,2}),*[-\s](\d{2,4})")
df

,Field1,day_month,month_day,year
0,He who sleeps on the floor will not fall off t...,16-APR,NaN,2005
1,"After all is said and done, more is said than ...",09-JAN,NaN,1856
2,I want to see you shoot the way you shoutTeddy...,NaN,Nov 16,1900
3,get someone else to do it.15-APR-1944This reco...,15-APR,NaN,1944
4,Why do they call it rush hour when nothing mov...,27-JUN,NaN,70
5,I'm taking the Ryanair approach to it: subcont...,23-MAY,NaN,2011
6,I Xeroxed a mirror. Now I have an extra Xerox...,30-JUN,NaN,06
7,Freidrich Engels01-AUG-08This record is missin...,01-AUG,NaN,08
8,'He's so old his social security number is two...,NaN,Jan 5,2000
9,"""I was the best man at the wedding.So why is s...",09-July,NaN,2001


In [ ]:
df["year"] = pd.to_numeric(df["year"])
df["year"] = np.where(
    df["year"] < 100, np.where(
        df["year"] < pd.Timestamp.today().year % 100,   # 26 for 2026
            df["year"] + 2000,                          # if number is under 26, 20xx
            df["year"] + 1900                           # if number is over 26, 19xx
    ), 
    df["year"]
)
df

,Field1,day_month,month_day,year
0,He who sleeps on the floor will not fall off t...,16-APR,NaN,2005
1,"After all is said and done, more is said than ...",09-JAN,NaN,1856
2,I want to see you shoot the way you shoutTeddy...,NaN,Nov 16,1900
3,get someone else to do it.15-APR-1944This reco...,15-APR,NaN,1944
4,Why do they call it rush hour when nothing mov...,27-JUN,NaN,1970
5,I'm taking the Ryanair approach to it: subcont...,23-MAY,NaN,2011
6,I Xeroxed a mirror. Now I have an extra Xerox...,30-JUN,NaN,2006
7,Freidrich Engels01-AUG-08This record is missin...,01-AUG,NaN,2008
8,'He's so old his social security number is two...,NaN,Jan 5,2000
9,"""I was the best man at the wedding.So why is s...",09-July,NaN,2001


In [ ]:
def get_day(row):
    if pd.isna(row["day_month"]):
        return row["month_day"].split(" ")[1]
    else:
        return row["day_month"].split("-")[0]

def get_month(row):
    if pd.isna(row["day_month"]):
        return row["month_day"].split(" ")[0][0:3] # only first 3 characters of month name
    else:
        return row["day_month"].split("-")[1][0:3]

df["day"] = df.apply(get_day, axis=1)
df["month"] = df.apply(get_month, axis=1)
df["month"] = pd.to_datetime(df.month, format='%b').dt.month # convert month to integer (requires 3-character month)
df

,Field1,day_month,month_day,year,day,month
0,He who sleeps on the floor will not fall off t...,16-APR,NaN,2005,16,4
1,"After all is said and done, more is said than ...",09-JAN,NaN,1856,09,1
2,I want to see you shoot the way you shoutTeddy...,NaN,Nov 16,1900,16,11
3,get someone else to do it.15-APR-1944This reco...,15-APR,NaN,1944,15,4
4,Why do they call it rush hour when nothing mov...,27-JUN,NaN,1970,27,6
5,I'm taking the Ryanair approach to it: subcont...,23-MAY,NaN,2011,23,5
6,I Xeroxed a mirror. Now I have an extra Xerox...,30-JUN,NaN,2006,30,6
7,Freidrich Engels01-AUG-08This record is missin...,01-AUG,NaN,2008,01,8
8,'He's so old his social security number is two...,NaN,Jan 5,2000,5,1
9,"""I was the best man at the wedding.So why is s...",09-July,NaN,2001,09,7


In [ ]:
df["Date"] = pd.to_datetime(df[["year", "month", "day"]]) # create date from year, month, day columns
df = df[["Field1", "Date"]]
df

,Field1,Date
0,He who sleeps on the floor will not fall off t...,2005-04-16
1,"After all is said and done, more is said than ...",1856-01-09
2,I want to see you shoot the way you shoutTeddy...,1900-11-16
3,get someone else to do it.15-APR-1944This reco...,1944-04-15
4,Why do they call it rush hour when nothing mov...,1970-06-27
5,I'm taking the Ryanair approach to it: subcont...,2011-05-23
6,I Xeroxed a mirror. Now I have an extra Xerox...,2006-06-30
7,Freidrich Engels01-AUG-08This record is missin...,2008-08-01
8,'He's so old his social security number is two...,2000-01-05
9,"""I was the best man at the wedding.So why is s...",2001-07-09


In [ ]:
df = df.rename(columns={'Date': 'DateTime_out'}) # align column names for comparison
print(df.eq(df_output))

    Field1  DateTime_out
0     True          True
1     True          True
2     True          True
3     True          True
4     True          True
5     True          True
6     True          True
7     True          True
8     True          True
9     True          True
10    True          True
11    True          True
12    True          True
13    True          True
14    True          True
15    True          True
16    True          True
